# 2.5 Optimal Trading Strategy

In this notebook, we convert the synthetic alpha signals from Section 2.4 into actual intraday trade schedules.

The previous sections provide:

$$
\text{prices},\quad
\text{fitted impact parameters},\quad
\text{backtest engine},\quad
\text{synthetic alpha signals}.
$$

The output of this notebook is a collection of trade panels

$$
q_{i,d,j},
$$

where:

| Symbol | Meaning |
|---|---|
| $i$ | stock |
| $d$ | trading date |
| $j$ | intraday time bin |
| $q_{i,d,j}>0$ | buy trade |
| $q_{i,d,j}<0$ | sell trade |

The core idea is:

$$
\alpha
\longrightarrow
\text{target impact}
\longrightarrow
\text{target impact state}
\longrightarrow
\text{trades}.
$$

We first construct optimal strategies under the fitted impact models, then compare them to benchmark schedules such as round-trip TWAP. We later extend the same structure to overnight alpha.

---

## 1. Common Inputs and Notation

For each stock-day, we use the following objects.

| Object | Meaning |
|---|---|
| $\alpha_{i,d,j}$ | synthetic alpha level |
| $\alpha'_{i,d,j}$ | alpha time derivative |
| $\beta_i$ | impact decay rate |
| $\Delta t$ | time-bin size |
| $a_i = e^{-\beta_i \Delta t}$ | one-step impact persistence |
| $\hat{\lambda}_i$ | fitted impact scale |
| $\sigma_i$ | stock volatility normalizer |
| $ADV_i$ | average daily volume normalizer |
| $q_{i,d,j}$ | strategy trade in shares |

In discrete time, the alpha derivative is approximated by

$$
\alpha'_{i,d,j}
\approx
\frac{
\alpha_{i,d,j} - \alpha_{i,d,j-1}
}{\Delta t}.
$$

Equivalently, if we define alpha decay as

$$
\text{decay}_{i,d,j}
=
-\alpha'_{i,d,j},
$$

then formulas involving

$$
\alpha - \beta^{-1}\alpha'
$$

can also be written as

$$
\alpha + \beta^{-1}\text{decay}.
$$

This convention must be kept consistent in the code.


---

## 2. OW Optimal Strategy

### 2.1 OW impact dynamics

Under the OW model, the normalized impact state follows

$$
\bar{I}_{i,d,j}^{OW}
=
a_i \bar{I}_{i,d,j-1}^{OW}
+
\sigma_i \frac{q_{i,d,j}}{ADV_i}.
$$

The fitted return impact is then

$$
I_{i,d,j}^{OW}
=
\hat{\lambda}_i
\bar{I}_{i,d,j}^{OW}.
$$

So the fitted model separates the impact into two parts:

| Part | Formula | Meaning |
|---|---|---|
| Normalized state | $\bar{I}^{OW}$ | impact state built from trades |
| Fitted impact | $I^{OW}=\hat{\lambda}\bar{I}^{OW}$ | return impact used in the strategy/backtest |

---

### 2.2 OW target impact

For a deterministic alpha process, the lecture target-impact rule is

$$
I_{i,d,j}^{*,OW}
=
\frac{1}{2}
\left(
\alpha_{i,d,j}
-
\frac{1}{\beta_i}\alpha'_{i,d,j}
\right).
$$

Using the decay convention,

$$
\text{decay}_{i,d,j}=-\alpha'_{i,d,j},
$$

this becomes

$$
I_{i,d,j}^{*,OW}
=
\frac{1}{2}
\left(
\alpha_{i,d,j}
+
\frac{1}{\beta_i}\text{decay}_{i,d,j}
\right).
$$

This target is in **return-impact space**, not share space.

---

### 2.3 Convert OW target impact into target state

Since

$$
I_{i,d,j}^{OW}
=
\hat{\lambda}_i
\bar{I}_{i,d,j}^{OW},
$$

the normalized target impact state is

$$
\bar{I}_{i,d,j}^{*,OW}
=
\frac{
I_{i,d,j}^{*,OW}
}{
\hat{\lambda}_i
}.
$$

---

### 2.4 Recover OW trades

The OW state recurrence is

$$
\bar{I}_{i,d,j}^{*,OW}
=
a_i \bar{I}_{i,d,j-1}^{*,OW}
+
\sigma_i \frac{q_{i,d,j}}{ADV_i}.
$$

Solving for trades gives

$$
q_{i,d,j}^{OW}
=
\frac{ADV_i}{\sigma_i}
\left(
\bar{I}_{i,d,j}^{*,OW}
-
a_i \bar{I}_{i,d,j-1}^{*,OW}
\right).
$$

Equivalently, substituting

$$
\bar{I}^{*,OW}=\frac{I^{*,OW}}{\hat{\lambda}},
$$

we get

$$
q_{i,d,j}^{OW}
=
\frac{ADV_i}{\hat{\lambda}_i\sigma_i}
\left(
I_{i,d,j}^{*,OW}
-
a_i I_{i,d,j-1}^{*,OW}
\right).
$$

---

## 3. AFS Optimal Strategy

### 3.1 AFS impact dynamics

The AFS model introduces a latent volume-space state

$$
J_{i,d,j}
=
a_i J_{i,d,j-1}
+
\sigma_i \frac{q_{i,d,j}}{ADV_i}.
$$

The normalized AFS impact feature is nonlinear:

$$
\bar{I}_{i,d,j}^{AFS}
=
\operatorname{sign}(J_{i,d,j})
|J_{i,d,j}|^c.
$$

For the square-root AFS model,

$$
c=\frac{1}{2},
$$

so

$$
\bar{I}_{i,d,j}^{AFS}
=
\operatorname{sign}(J_{i,d,j})
\sqrt{|J_{i,d,j}|}.
$$

The fitted return impact is

$$
I_{i,d,j}^{AFS}
=
\hat{\lambda}_i
\operatorname{sign}(J_{i,d,j})
|J_{i,d,j}|^c.
$$

---

### 3.2 AFS target impact

For AFS, the lecture target-impact rule is

$$
I_{i,d,j}^{*,AFS}
=
\frac{1}{1+c}
\left(
\alpha_{i,d,j}
-
\frac{1}{\beta_i}\alpha'_{i,d,j}
\right).
$$

For square-root AFS,

$$
c=\frac{1}{2},
$$

therefore

$$
I_{i,d,j}^{*,AFS}
=
\frac{2}{3}
\left(
\alpha_{i,d,j}
-
\frac{1}{\beta_i}\alpha'_{i,d,j}
\right).
$$

Using the decay convention,

$$
\text{decay}_{i,d,j}=-\alpha'_{i,d,j},
$$

this becomes

$$
I_{i,d,j}^{*,AFS}
=
\frac{2}{3}
\left(
\alpha_{i,d,j}
+
\frac{1}{\beta_i}\text{decay}_{i,d,j}
\right).
$$

---

### 3.3 Convert AFS target impact into target $J$ state

The fitted AFS impact is

$$
I_{i,d,j}^{AFS}
=
\hat{\lambda}_i
\operatorname{sign}(J_{i,d,j})
|J_{i,d,j}|^c.
$$

First divide by the fitted impact scale:

$$
\bar{I}_{i,d,j}^{*,AFS}
=
\frac{
I_{i,d,j}^{*,AFS}
}{
\hat{\lambda}_i
}.
$$

Then invert the nonlinear impact map:

$$
J_{i,d,j}^{*}
=
\operatorname{sign}
\left(
\bar{I}_{i,d,j}^{*,AFS}
\right)
\left|
\bar{I}_{i,d,j}^{*,AFS}
\right|^{1/c}.
$$

For square-root AFS, where $c=1/2$,

$$
J_{i,d,j}^{*}
=
\operatorname{sign}
\left(
\bar{I}_{i,d,j}^{*,AFS}
\right)
\left|
\bar{I}_{i,d,j}^{*,AFS}
\right|^{2}.
$$

---

### 3.4 Recover AFS trades

The target $J$ state satisfies

$$
J_{i,d,j}^{*}
=
a_i J_{i,d,j-1}^{*}
+
\sigma_i \frac{q_{i,d,j}}{ADV_i}.
$$

Solving for trades gives

$$
q_{i,d,j}^{AFS}
=
\frac{ADV_i}{\sigma_i}
\left(
J_{i,d,j}^{*}
-
a_i J_{i,d,j-1}^{*}
\right).
$$

So for AFS, the full flow is

$$
\alpha
\longrightarrow
\alpha'
\longrightarrow
I^{*,AFS}
\longrightarrow
\bar{I}^{*,AFS}
\longrightarrow
J^*
\longrightarrow
q^{AFS}.
$$

---




## 4. Comparison of OW and AFS Strategy Construction

| Step | OW | AFS |
|---|---|---|
| Latent state | $\bar{I}^{OW}$ | $J$ |
| State recurrence | $\bar{I}_j=a\bar{I}_{j-1}+\sigma q_j/ADV$ | $J_j=aJ_{j-1}+\sigma q_j/ADV$ |
| Impact map | $I=\hat{\lambda}\bar{I}$ | $I=\hat{\lambda}\operatorname{sign}(J)|J|^c$ |
| Target impact | $\frac{1}{2}(\alpha-\beta^{-1}\alpha')$ | $\frac{1}{1+c}(\alpha-\beta^{-1}\alpha')$ |
| Square-root case | not applicable | $c=1/2$, so coefficient is $2/3$ |
| Inversion | linear | nonlinear |
| Trade recovery | from target $\bar{I}^*$ | from target $J^*$ |

The important difference is that OW is linear all the way from impact to trades. AFS keeps a linear state recurrence, but the map from state to impact is nonlinear. Therefore, AFS requires one extra inversion step:

$$
I^* \rightarrow J^*.
$$

---

## 5. Strategy Objects Built in This Notebook

The notebook constructs trade panels for several strategies.

| Strategy | Description |
|---|---|
| `ow_intraday_only` | OW optimal strategy using intraday synthetic alpha |
| `afs_intraday_only` | AFS optimal strategy using intraday synthetic alpha |
| `reduced_intraday_only` | Reduced optimal strategy using intraday synthetic alpha |

Each strategy produces a trade DataFrame with the same structure:

| Component | Format |
|---|---|
| Index | `(stock, date)` |
| Columns | intraday time bins |
| Values | trades $q_{i,d,j}$ in shares |

This common format allows every strategy to be passed into the same backtest engine.

---

## 6. Implementation Summary

The implementation follows the same structure for each model.

| Step | Operation |
|---|---|
| 1 | Load fitted model parameters |
| 2 | Load synthetic alpha panel |
| 3 | Compute alpha derivative or alpha decay |
| 4 | Compute target impact $I^*$ |
| 5 | Convert target impact into the model-specific target state |
| 6 | Recover trades from the target state recurrence |

The final output is a dictionary of trade panels:

$$
\{\text{strategy name} \mapsto q_{i,d,j}\}.
$$

This dictionary is then used in the performance-reporting section to compare P&L, transaction costs, drawdowns, and impact dislocations across strategies.

# Reduced-form dynamic-liquidity optimal strategy

In addition to the OW and AFS optimal strategies, we also implement an optimal strategy for the reduced-form dynamic-liquidity model. The goal of this model is to make the price impact per share depend on the amount of local market activity. Intuitively, if the market is currently trading a lot, then our order should have lower impact per share. If the market is quiet, the same order should have larger impact per share.

This makes the reduced-form model more adaptive than the standard OW model. In OW, the impact of a trade is scaled by the stock-level ADV and volatility, but the impact coefficient is constant throughout the day. In the reduced-form model, the impact coefficient changes intraday through a local volume state.

---

## 1. Reduced-form impact model

The reduced-form model uses the impact recursion

$$
I_t
=
e^{-\beta \Delta t} I_{t-\Delta t}
+
\lambda \sigma
\frac{q_t}{\sqrt{ADV \cdot v_t}}.
$$

Here:

- $I_t$ is the price impact state at time $t$,
- $q_t$ is the strategy trade at time $t$,
- $\lambda$ is the fitted impact coefficient,
- $\sigma$ is the stock-level volatility estimate,
- $ADV$ is the stock-level average daily volume,
- $\beta$ is the impact decay rate,
- $\Delta t$ is the time step, equal to 10 seconds in our data,
- $v_t$ is the local market-volume state.

The decay rate is computed from the half-life $H$:

$$
\beta = \frac{\log(2)}{H}.
$$

For example, if the half-life is 1 hour, then

$$
H = 3600 \text{ seconds},
$$

so

$$
\beta = \frac{\log(2)}{3600}.
$$

The term

$$
e^{-\beta \Delta t}
$$

controls how much impact remains from one 10-second bin to the next.

---

## 2. Local volume state

The main difference between OW and the reduced-form model is the local volume state $v_t$.

We compute $v_t$ from the public tape volume using an exponential moving average of unsigned public traded volume:

$$
v_t
=
e^{-\beta \Delta t}v_{t-\Delta t}
+
|q_t^{public}|.
$$

This is a decaying measure of recent market activity. If there has been a lot of recent public trading volume, $v_t$ is high. If the market has been quiet, $v_t$ is low.

The reduced-form impact input is then scaled by

$$
\sqrt{ADV \cdot v_t}.
$$

So the impact per share is approximately proportional to

$$
\frac{1}{\sqrt{v_t}}.
$$

This means:

- when $v_t$ is high, market liquidity is high and impact per share is lower;
- when $v_t$ is low, market liquidity is low and impact per share is higher.

Therefore, the reduced-form model encourages the strategy to trade more aggressively during high-volume periods and more cautiously during low-volume periods.

---

## 3. Relationship with OW

The OW model has the impact recursion

$$
I_t
=
e^{-\beta \Delta t} I_{t-\Delta t}
+
\lambda \sigma \frac{q_t}{ADV}.
$$

In OW, the scaling term is constant through the day:

$$
\lambda \sigma / ADV.
$$

In the reduced-form model, the scaling term is time-varying:

$$
\lambda_t
=
\frac{\lambda \sigma}{\sqrt{ADV \cdot v_t}}.
$$

So OW assumes a constant impact per share, while the reduced-form model assumes that impact per share varies with local market volume.

The reduced-form model can therefore be seen as an OW-style model with dynamic liquidity.

---

## 4. Dynamic-liquidity target impact

As with OW and AFS, we first compute an optimal target impact path $I_t^*$, and then recover the trades needed to track this target impact.

For OW, the optimal target impact is

$$
I_t^*
=
\frac{1}{2}
\left(
\alpha_t
-
\frac{\mu_t}{\beta}
\right),
$$

where:

- $\alpha_t$ is the alpha level,
- $\mu_t$ is the alpha drift,
- $\mu_t$ is approximated by

$$
\mu_t
\approx
\frac{\alpha_{t+\Delta t} - \alpha_t}{\Delta t}.
$$

For the reduced-form model, liquidity changes through time. We define the time-varying impact coefficient as

$$
\lambda_t
=
\frac{\lambda}{\sqrt{v_t}}.
$$

Taking logs,

$$
\gamma_t
=
\log(\lambda_t)
=
\text{constant}
-
\frac{1}{2}\log(v_t).
$$

Therefore,

$$
\gamma'_t
=
-\frac{1}{2}
\frac{d \log(v_t)}{dt}.
$$

The dynamic-liquidity optimal target impact is

$$
I_t^*
=
\frac{\beta+\gamma'_t}{2\beta+\gamma'_t}\alpha_t
-
\frac{1}{2\beta+\gamma'_t}\mu_t.
$$

This is the target impact used when `use_dynamic_target=True`.

The term $\gamma'_t$ adjusts the target impact for changes in local liquidity. If local liquidity is changing quickly, the optimal target can differ from the OW target. If liquidity is changing slowly, $\gamma'_t$ is small and the formula approximately reduces to the OW target.

---

## 5. Slow-liquidity approximation

When local liquidity changes slowly, we have

$$
\gamma'_t \approx 0.
$$

Then the reduced-form target becomes

$$
I_t^*
\approx
\frac{\beta}{2\beta}\alpha_t
-
\frac{1}{2\beta}\mu_t.
$$

So

$$
I_t^*
\approx
\frac{1}{2}
\left(
\alpha_t
-
\frac{\mu_t}{\beta}
\right).
$$

This is exactly the OW target impact.

## 1. Imports and paths

This notebook assumes that the helper `.py` files and the saved results from Sections 2.1--2.4 are available in the same folder as this notebook.

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.backtest_engine import get_best_model_fit_df
from src.optimal_trading_strategies import *

DATA_DIR = Path("data")

DT_SECONDS = 10
TARGET_PARTICIPATION = 0.20
NORMALIZE_ABS_VOLUME = True

TRAIN_MONTH = "201901"
TEST_MONTH = "201902"
N_STOCKS = 20

In [24]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import src.data_prep as data_prep
import src.backtest_engine as backtest_engine
import src.optimal_trading_strategies as optimal_trading_strategies
import src.performance_metrics as performance_metrics
import importlib

DATA_DIR = r"data"

DATA_DIR = Path(DATA_DIR) 

importlib.reload(data_prep)
importlib.reload(backtest_engine)
importlib.reload(optimal_trading_strategies)
importlib.reload(performance_metrics)

from src.data_prep import *
from src.backtest_engine import *
from src.optimal_trading_strategies import *
from src.performance_metrics import *



## 2. Load inputs

In [25]:
# %%
test_px_df = load_panel_csv(
    DATA_DIR / "test_px_201902_20.csv"
)

test_traded_volume_df = load_panel_csv(
    DATA_DIR / "test_traded_volume_201902_20.csv"
)

scaling_df = load_stock_level_csv(
    DATA_DIR / "scaling_201901_20.csv"
)

synthetic_alpha_df = load_panel_csv(
    DATA_DIR / "synthetic_alpha_201902_20.csv"
)

stock_results_df = pd.read_csv(
    DATA_DIR / "impact_model_stock_results_201901_train_201902_test.csv"
)

best_df = pd.read_csv(
    DATA_DIR / "impact_model_best_by_is_201901_train_201902_test.csv"
)

## Create fitted parameter dictionary

In [26]:
# %%
ow_fit_df = get_best_model_fit_df(
    stock_results_df=stock_results_df,
    best_df=best_df,
    model_type="ow",
)

afs_fit_df = get_best_model_fit_df(
    stock_results_df=stock_results_df,
    best_df=best_df,
    model_type="afs",
)

reduced_fit_df = get_best_model_fit_df(
    stock_results_df=stock_results_df,
    best_df=best_df,
    model_type="reduced_form",
)

fit_dfs = {
    "ow": ow_fit_df,
    "afs": afs_fit_df,
    "reduced_form": reduced_fit_df,
}

In [27]:
# %%
for model_name, fit_df in fit_dfs.items():
    print(model_name, fit_df.shape)
    print(fit_df[["lambda_hat", "half_life_seconds", "is_r2", "oos_r2"]].head())

ow (20, 24)
       lambda_hat  half_life_seconds     is_r2    oos_r2
stock                                                   
AAL    262.866026               1800  0.143934  0.199819
AAPL   407.392206               1800  0.231705  0.252164
ABBV   382.822089               1800  0.130716  0.131900
ABT    440.867448               1800  0.171972  0.115936
ADBE   339.189338               1800  0.123732  0.137870
afs (20, 24)
       lambda_hat  half_life_seconds     is_r2    oos_r2
stock                                                   
AAL      0.770608                300  0.151269  0.185629
AAPL     0.828313                300  0.234038  0.227081
ABBV     0.540337                300  0.102414  0.115905
ABT      0.550386                300  0.133748  0.099200
ADBE     0.573366                300  0.130973  0.123505
reduced_form (20, 24)
       lambda_hat  half_life_seconds     is_r2    oos_r2
stock                                                   
AAL     73.752339                600  0.1

### Create alpha dictionary

In [28]:
alpha_dfs = {
    "intraday_only": synthetic_alpha_df,
}

## Construct OW day-only strategy

In [29]:
# %%
(
    ow_day_only_trades_df,
    ow_day_only_target_impact_df,
    ow_day_only_alpha_mu_df,
    ow_day_only_scale_df,
) = make_ow_optimal_trade_df(
    alpha_df=alpha_dfs["intraday_only"],
    test_px_df=test_px_df,
    scaling_df=scaling_df,
    fit_df=fit_dfs["ow"],
    dt_seconds=DT_SECONDS,
    normalize_abs_volume=NORMALIZE_ABS_VOLUME,
    target_participation=TARGET_PARTICIPATION,
)

In [30]:
# %%
print("ow_day_only_trades_df:", ow_day_only_trades_df.shape)
print("ow_day_only_target_impact_df:", ow_day_only_target_impact_df.shape)
print("ow_day_only_alpha_mu_df:", ow_day_only_alpha_mu_df.shape)
print("ow_day_only_scale_df:", ow_day_only_scale_df.shape)

ow_day_only_trades_df: (380, 2341)
ow_day_only_target_impact_df: (380, 2341)
ow_day_only_alpha_mu_df: (380, 2341)
ow_day_only_scale_df: (380, 14)


## Construct AFS day-only strategy

In [31]:
# %%
(
    afs_day_only_trades_df,
    afs_day_only_target_impact_df,
    afs_day_only_alpha_mu_df,
    afs_day_only_scale_df,
) = make_afs_optimal_trade_df(
    alpha_df=alpha_dfs["intraday_only"],
    test_px_df=test_px_df,
    scaling_df=scaling_df,
    fit_df=fit_dfs["afs"],
    dt_seconds=DT_SECONDS,
    c=0.5,
    normalize_abs_volume=NORMALIZE_ABS_VOLUME,
    target_participation=TARGET_PARTICIPATION,
    apply_terminal_condition=True,
)

In [32]:
# %%
print("afs_day_only_trades_df:", afs_day_only_trades_df.shape)
print("afs_day_only_target_impact_df:", afs_day_only_target_impact_df.shape)
print("afs_day_only_alpha_mu_df:", afs_day_only_alpha_mu_df.shape)
print("afs_day_only_scale_df:", afs_day_only_scale_df.shape)

afs_day_only_trades_df: (380, 2341)
afs_day_only_target_impact_df: (380, 2341)
afs_day_only_alpha_mu_df: (380, 2341)
afs_day_only_scale_df: (380, 16)


## Construct reduced-form day-only strategy

In [33]:
# %%
(
    reduced_day_only_trades_df,
    reduced_day_only_target_impact_df,
    reduced_day_only_alpha_mu_df,
    reduced_day_only_local_volume_df,
    reduced_day_only_gamma_prime_df,
    reduced_day_only_scale_df,
) = make_reduced_form_optimal_trade_df(
    alpha_df=alpha_dfs["intraday_only"],
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_df=fit_dfs["reduced_form"],
    dt_seconds=DT_SECONDS,
    normalize_abs_volume=NORMALIZE_ABS_VOLUME,
    target_participation=TARGET_PARTICIPATION,
    use_dynamic_target=True,
    gamma_method="log_backward",
    gamma_clip_multiple=0.5,
)

In [34]:
print("reduced_day_only_trades_df:", reduced_day_only_trades_df.shape)
print("reduced_day_only_target_impact_df:", reduced_day_only_target_impact_df.shape)
print("reduced_day_only_alpha_mu_df:", reduced_day_only_alpha_mu_df.shape)
print("reduced_day_only_local_volume_df:", reduced_day_only_local_volume_df.shape)
print("reduced_day_only_gamma_prime_df:", reduced_day_only_gamma_prime_df.shape)
print("reduced_day_only_scale_df:", reduced_day_only_scale_df.shape)

reduced_day_only_trades_df: (380, 2341)
reduced_day_only_target_impact_df: (380, 2341)
reduced_day_only_alpha_mu_df: (380, 2341)
reduced_day_only_local_volume_df: (380, 2341)
reduced_day_only_gamma_prime_df: (380, 2341)
reduced_day_only_scale_df: (380, 21)


## Store output in dicts

In [35]:
# %%
non_rolling_strategy_outputs = {
    "non_rolling_ow_day_only": {
        "model_type": "ow",
        "alpha_key": "intraday_only",
        "trades": ow_day_only_trades_df,
        "target_impact": ow_day_only_target_impact_df,
        "alpha_mu": ow_day_only_alpha_mu_df,
        "scale": ow_day_only_scale_df,
    },

    "non_rolling_afs_day_only": {
        "model_type": "afs",
        "alpha_key": "intraday_only",
        "trades": afs_day_only_trades_df,
        "target_impact": afs_day_only_target_impact_df,
        "alpha_mu": afs_day_only_alpha_mu_df,
        "scale": afs_day_only_scale_df,
    },

    "non_rolling_reduced_day_only": {
        "model_type": "reduced_form",
        "alpha_key": "intraday_only",
        "trades": reduced_day_only_trades_df,
        "target_impact": reduced_day_only_target_impact_df,
        "alpha_mu": reduced_day_only_alpha_mu_df,
        "local_volume": reduced_day_only_local_volume_df,
        "gamma_prime": reduced_day_only_gamma_prime_df,
        "scale": reduced_day_only_scale_df,
    },
}

In [36]:
# %%
non_rolling_strategy_trade_dfs = {
    strategy_name: strategy_data["trades"]
    for strategy_name, strategy_data in non_rolling_strategy_outputs.items()
}

In [37]:
# %%
for strategy_name, trades_df in non_rolling_strategy_trade_dfs.items():
    print(strategy_name, trades_df.shape)

non_rolling_ow_day_only (380, 2341)
non_rolling_afs_day_only (380, 2341)
non_rolling_reduced_day_only (380, 2341)


## Validation checks

In [38]:
# %%
for strategy_name, strategy_data in non_rolling_strategy_outputs.items():

    trades_df = strategy_data["trades"]
    target_impact_df = strategy_data["target_impact"]
    alpha_mu_df = strategy_data["alpha_mu"]
    scale_df = strategy_data["scale"]

    assert trades_df.shape == test_px_df.shape
    assert target_impact_df.shape == test_px_df.shape
    assert alpha_mu_df.shape == test_px_df.shape

    assert trades_df.index.equals(test_px_df.index)
    assert target_impact_df.index.equals(test_px_df.index)
    assert alpha_mu_df.index.equals(test_px_df.index)

    assert list(trades_df.columns) == list(test_px_df.columns)
    assert list(target_impact_df.columns) == list(test_px_df.columns)
    assert list(alpha_mu_df.columns) == list(test_px_df.columns)

    assert scale_df.shape[0] > 0

print("All strategy output checks passed.")

All strategy output checks passed.


In [168]:
# %%
strategy_volume_check_df = []

for strategy_name, strategy_data in non_rolling_strategy_outputs.items():
    scale_df = strategy_data["scale"].copy()
    scale_df["strategy"] = strategy_name

    strategy_volume_check_df.append(scale_df)

strategy_volume_check_df = pd.concat(strategy_volume_check_df, ignore_index=True)

strategy_volume_check_df.groupby("strategy")[
    ["target_abs_volume", "raw_abs_volume", "actual_abs_volume", "net_traded", "scale_factor"]
].agg(["mean", "median", "std"]).round(4)

target_abs_volume                          \
                                          mean      median         std   
strategy                                                                 
non_rolling_afs_day_only            210496.521  86099.1952  395491.993   
non_rolling_ow_day_only             210496.521  86099.1952  395491.993   
non_rolling_reduced_day_only        210496.521  86099.1952  395491.993   

                             raw_abs_volume                              \
                                       mean        median           std   
strategy                                                                  
non_rolling_afs_day_only       2.761847e+07  1.845678e+07  2.895720e+07   
non_rolling_ow_day_only        9.120921e+07  4.069491e+07  1.454286e+08   
non_rolling_reduced_day_only   1.942843e+07  9.519752e+06  3.075296e+07   

                             actual_abs_volume                          \
                                          mean      median         std   
strategy                                                                 
non_rolling_afs_day_only            210496.521  86099.1952  395491.993   
non_rolling_ow_day_only             210496.521  86099.1952  395491.993   
non_rolling_reduced_day_only        210496.521  86099.1952  395491.993   

                             net_traded                     scale_factor  \
                                   mean   median        std         mean   
strategy                                                                   
non_rolling_afs_day_only       161.5644  66.6937  5421.9772       0.0060   
non_rolling_ow_day_only          7.4712   2.8663   204.7195       0.0021   
non_rolling_reduced_day_only    61.6001  21.9230  1708.7634       0.0095   

                                              
                              median     std  
strategy                                      
non_rolling_afs_day_only      0.0051  0.0036  
non_rolling_ow_day_only       0.0020  0.0005  
non_rolling_reduced_day_only  0.0091  0.0025

## Saving files

In [39]:
# ------------------------------------------------------------
# Save final 2.5 strategy outputs needed for later metrics
# ------------------------------------------------------------

OUTPUT_DIR = DATA_DIR

files_to_save = {
    # OW
    "strategy_trades_non_rolling_ow_day_only_201902_20.csv": ow_day_only_trades_df,
    "strategy_target_impact_non_rolling_ow_day_only_201902_20.csv": ow_day_only_target_impact_df,

    # AFS
    "strategy_trades_non_rolling_afs_day_only_201902_20.csv": afs_day_only_trades_df,
    "strategy_target_impact_non_rolling_afs_day_only_201902_20.csv": afs_day_only_target_impact_df,

    # Reduced-form
    "strategy_trades_non_rolling_reduced_day_only_201902_20.csv": reduced_day_only_trades_df,
    "strategy_target_impact_non_rolling_reduced_day_only_201902_20.csv": reduced_day_only_target_impact_df,
}

for filename, df in files_to_save.items():
    df.reset_index().to_csv(OUTPUT_DIR / filename, index=False)

print("Saved files:")
for filename in files_to_save:
    print(" -", filename)

Saved files:
 - strategy_trades_non_rolling_ow_day_only_201902_20.csv
 - strategy_target_impact_non_rolling_ow_day_only_201902_20.csv
 - strategy_trades_non_rolling_afs_day_only_201902_20.csv
 - strategy_target_impact_non_rolling_afs_day_only_201902_20.csv
 - strategy_trades_non_rolling_reduced_day_only_201902_20.csv
 - strategy_target_impact_non_rolling_reduced_day_only_201902_20.csv
